# Bài 16: Wavelet và Bộ lọc Gabor (Wavelets and Gabor Filters)

Biến đổi Fourier (Bài 15) cho bạn biết *những* tần số nào có mặt trong tín hiệu, nhưng không cho biết *ở đâu* &mdash; một hàm cơ sở sóng sin trải dài trên toàn bộ ảnh, nên một đặc trưng cục bộ (một biên, một mảng kết cấu) bị nhòe ra trên toàn phổ. **Wavelet** và **bộ lọc Gabor (Gabor filters)** là hai cách khắc phục khác nhau cho vấn đề này: cả hai đều được xây dựng từ các dao động nhỏ, cục bộ về không gian thay vì các hình sin vô hạn, mang lại cảm nhận đồng thời về *ở đâu* và *tần số nào*.

In [ ]:
import numpy as np
import cv2
import pywt
import matplotlib.pyplot as plt

## Wavelet Haar: wavelet đơn giản nhất có thể (The Haar wavelet: the simplest possible wavelet)
Biến đổi wavelet Haar tách một tín hiệu thành phần **xấp xỉ (approximation)** (các trung bình cục bộ) và phần **chi tiết (detail)** (các hiệu cục bộ), tính trên các cặp mẫu không chồng lấn:

$$a_k = \frac{x_{2k} + x_{2k+1}}{\sqrt{2}}, \qquad d_k = \frac{x_{2k} - x_{2k+1}}{\sqrt{2}}$$

Điều này hẳn trông quen thuộc: về bản chất nó là cùng ý tưởng "làm mờ + giữ phần dư" như kim tự tháp Laplace (Bài 13), chỉ là tính theo từng cặp thay vì bằng kernel Gauss, và không có chồng lấn giữa các vùng lân cận.

In [ ]:
signal = np.array([1, 3, 5, 11, 7, 9, 2, 4], dtype=np.float64)

approx = (signal[0::2] + signal[1::2]) / np.sqrt(2)
detail = (signal[0::2] - signal[1::2]) / np.sqrt(2)

cA_ref, cD_ref = pywt.dwt(signal, 'haar')
print('manual matches pywt.dwt?', np.allclose(approx, cA_ref) and np.allclose(detail, cD_ref))
print('approximation (cA):', np.round(approx, 2))
print('detail (cD):       ', np.round(detail, 2))

Ảnh động này minh họa quá trình tính biến đổi Haar của một tín hiệu qua 3 tầng. Các kernel Haar 2 phần tử (trung bình và hiệu) trượt qua đầu vào của tầng hiện tại mỗi lần hai mẫu, tạo ra một giá trị xấp xỉ (xanh lục) và một giá trị chi tiết (xám) ở mỗi bước. Phần xấp xỉ sau đó trở thành đầu vào cho tầng tiếp theo có độ dài bằng một nửa — phiên bản 1D của các chuỗi kim tự tháp ở Bài 11 và 13.

<img src="../animations/haar_cascade_demo.gif" alt="Ảnh động chuỗi wavelet Haar 3 tầng: một bộ lọc 2 phần tử trượt qua tín hiệu của mỗi tầng tạo ra giá trị xấp xỉ và chi tiết theo từng cặp, với phần xấp xỉ đệ quy vào tầng tiếp theo" style="display: block; margin: 0 auto; max-width: 680px;">

## Từ Haar đến Daubechies: các wavelet mượt hơn (From Haar to Daubechies: smoother wavelets)
Wavelet Haar không liên tục (một bậc cứng), khiến nó có khả năng định vị tần số kém &mdash; nội dung tần số của chính nó bị trải rộng, ngược với điều ta muốn. **Wavelet Daubechies** (<a href="../references.html#daubechies-1988">Daubechies, 1988</a>) dùng các bộ lọc dài hơn, mượt hơn với nhiều *mô-men triệt tiêu (vanishing moments)* hơn, đánh đổi phạm vi không gian rộng hơn để có hành vi tần số tốt hơn nhiều. `db2` (đôi khi gọi là "D4" vì có 4 phần tử bộ lọc) là bước tiếp theo về độ mượt so với Haar.

In [ ]:
db2 = pywt.Wavelet('db2')
print('db2 low-pass (scaling) filter :', np.round(db2.dec_lo, 4))
print('db2 high-pass (wavelet) filter:', np.round(db2.dec_hi, 4))

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].stem(db2.dec_lo)
axes[0].set_title('db2 scaling filter (low-pass)')
axes[1].stem(db2.dec_hi)
axes[1].set_title('db2 wavelet filter (high-pass)')
plt.tight_layout()
plt.show()

Biến đổi Haar ở trên chỉ là một phép tích-chập-rồi-giảm-mẫu-2-lần với các bộ lọc 2 phần tử $[\tfrac{1}{\sqrt2}, \tfrac{1}{\sqrt2}]$ và $[\tfrac{1}{\sqrt2}, -\tfrac{1}{\sqrt2}]$. Wavelet Daubechies theo đúng công thức đó nhưng với các bộ lọc dài hơn. Ta tái tạo phép phân rã của `db2` từ đầu, dưới dạng tích chập tuần hoàn, và đối chiếu với `pywt`.

In [ ]:
def dwt_periodic(x, filt):
    """One level of a periodic discrete wavelet transform with an arbitrary filter."""
    n, L = len(x), len(filt)
    k = np.arange(n // 2)[:, None]
    i = np.arange(L)[None, :]
    idx = (2 * k + 2 - i) % n
    return (filt[None, :] * x[idx]).sum(axis=1)

longer_signal = np.array([1, 3, 5, 11, 7, 9, 2, 4, 6, 8], dtype=np.float64)

cA_mine = dwt_periodic(longer_signal, np.array(db2.dec_lo))
cD_mine = dwt_periodic(longer_signal, np.array(db2.dec_hi))
cA_ref, cD_ref = pywt.dwt(longer_signal, db2, mode='periodization')

print('matches pywt (periodization mode)?', np.allclose(cA_mine, cA_ref) and np.allclose(cD_mine, cD_ref))

reconstructed = pywt.idwt(cA_mine, cD_mine, db2, mode='periodization')
print('perfect reconstruction?', np.allclose(reconstructed, longer_signal))

## Phân rã wavelet 2D của một ảnh (2D wavelet decomposition of an image)
Giống như bộ lọc Gauss khả tách ở Bài 10, biến đổi wavelet 2D được áp dụng thành hai lượt 1D: theo hàng, rồi theo cột. Một tầng phân rã tách ảnh thành **bốn dải con (subband)**:

- **LL**: thông thấp theo cả hai phương &mdash; một bản sao thô hơn, độ phân giải một nửa của ảnh (giống một tầng kim tự tháp Gauss)
- **LH**: thông thấp theo hàng, thông cao theo cột &mdash; phản hồi với các biên ngang
- **HL**: thông cao theo hàng, thông thấp theo cột &mdash; phản hồi với các biên dọc
- **HH**: thông cao theo cả hai phương &mdash; phản hồi với chi tiết chéo và các góc

In [ ]:
img = np.zeros((200, 200), dtype=np.float64)
cv2.rectangle(img, (40, 40), (160, 160), 200, -1)
cv2.circle(img, (100, 100), 40, 100, -1)

LL, (LH, HL, HH) = pywt.dwt2(img, 'db2', mode='periodization')

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, im, title in zip(axes, [img, LL, LH, HL, HH],
                          ['Original', 'LL (approx.)', 'LH (horiz. edges)', 'HL (vert. edges)', 'HH (diagonal)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Cũng như kim tự tháp Laplace, phép phân rã này khả nghịch chính xác: `pywt.idwt2` tái tạo ảnh gốc từ bốn dải con mà không mất mát gì.

In [ ]:
reconstructed_img = pywt.idwt2((LL, (LH, HL, HH)), 'db2', mode='periodization')
print('max reconstruction error:', np.abs(reconstructed_img - img).max())

## Bộ lọc Gabor: dao động cục bộ được điều chỉnh theo hướng (Gabor filters: localized oscillations tuned to orientation)
**Bộ lọc Gabor** là một vân sóng hình sin nhân với một đường bao Gauss &mdash; một sóng cục bộ trong không gian, được điều chỉnh theo một tần số *và* một hướng cụ thể. Khác với wavelet Daubechies (được xây dựng cho phép phân rã đa độ phân giải gọn, trực giao, khả nghịch), bộ lọc Gabor được dùng nhiều hơn cho *trích xuất đặc trưng (feature extraction)*: phát hiện kết cấu và biên có hướng ở một tỉ lệ được chọn.

Bộ lọc Gabor cũng có một mối liên hệ sinh học đáng chú ý. Khi ghi lại hoạt động của các nơ-ron trong vỏ não thị giác của mèo, Hubel và Wiesel (<a href="../references.html#hubel-wiesel-1959">Hubel and Wiesel, 1959</a>) đã tìm thấy các "tế bào đơn giản (simple cells)" phát xung có chọn lọc với một thanh hoặc một biên ở một hướng và vị trí cụ thể, và hầu như không phát xung với các hướng khác &mdash; một khám phá nền tảng trong khoa học thần kinh thị giác. Về sau người ta chỉ ra (<a href="../references.html#marcelja-1980">Marcelja, 1980</a>; <a href="../references.html#daugman-1985">Daugman, 1985</a>) rằng hàm Gabor 2D là một mô hình toán học tốt đáng kinh ngạc cho trường tiếp nhận (receptive field) của các tế bào đơn giản này, và đó là một phần lý do bộ lọc Gabor trở thành công cụ chuẩn, có cảm hứng sinh học, để trích xuất đặc trưng chọn lọc theo hướng trong thị giác máy tính.

In [ ]:
orientations_deg = [0, 45, 90, 135]
wavelengths = [4, 8, 12, 16]

fig, axes = plt.subplots(len(wavelengths), len(orientations_deg), figsize=(11, 11))
for i, lambd in enumerate(wavelengths):
    for j, t in enumerate(orientations_deg):
        kernel = cv2.getGaborKernel((25, 25), sigma=3, theta=np.radians(t), lambd=lambd, gamma=0.5, psi=0)
        ax = axes[i, j]
        ax.imshow(kernel, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(fr'$\theta$ = {t} deg', fontsize=9)
        if j == 0:
            ax.set_ylabel(fr'$\lambda$ = {lambd}', fontsize=9)
plt.tight_layout()
plt.show()

### Tính chọn lọc theo hướng, minh họa định lượng (Orientation selectivity, demonstrated quantitatively)
Ta vẽ bốn thanh ở bốn hướng khác nhau và lọc ảnh bằng một kernel Gabor được điều chỉnh theo từng hướng, rồi đo độ lớn phản hồi trung bình gần mỗi thanh. Một bộ lọc phải phản hồi mạnh nhất với thanh khớp với hướng điều chỉnh của nó và phản hồi yếu với các thanh khác.

In [ ]:
def draw_bar(image, center, angle_deg, length=60, thickness=6, value=255):
    angle = np.radians(angle_deg)
    dx, dy = length / 2 * np.cos(angle), length / 2 * np.sin(angle)
    p1 = (int(center[0] - dx), int(center[1] - dy))
    p2 = (int(center[0] + dx), int(center[1] + dy))
    cv2.line(image, p1, p2, value, thickness)

bar_orientations = [0, 45, 90, 135]
centers = [(50, 50), (150, 50), (50, 150), (150, 150)]

bars_img = np.zeros((200, 200), dtype=np.float64)
for c, ang in zip(centers, bar_orientations):
    draw_bar(bars_img, c, ang)

plt.imshow(bars_img, cmap='gray')
plt.title('Bars at 0, 45, 90, 135 degrees')
plt.axis('off')
plt.show()

In [ ]:
responses = np.zeros((4, 4))
for fi, bar_angle in enumerate(bar_orientations):
    # Note: cv2's theta parameter is the orientation of the sinusoidal stripes inside the
    # kernel, which runs *perpendicular* to the bar it responds to -- hence the +90 offset.
    kernel = cv2.getGaborKernel((25, 25), sigma=4, theta=np.radians(bar_angle + 90), lambd=10, gamma=0.5, psi=0)
    response = cv2.filter2D(bars_img, cv2.CV_64F, kernel)
    for ci, c in enumerate(centers):
        region = response[c[1] - 20:c[1] + 20, c[0] - 20:c[0] + 20]
        responses[fi, ci] = np.abs(region).mean()

print(f'{"filter tuned for":>18}', ' '.join(f'{a:>6}' for a in bar_orientations), '  <- bar orientation')
for fi, ang in enumerate(bar_orientations):
    print(f'{ang:>18}', ' '.join(f'{v:>6.0f}' for v in responses[fi]))

best_match = [bar_orientations[i] for i in responses.argmax(axis=1)]
print('\neach filter peaks at its own bar orientation?', best_match == bar_orientations)

Ảnh động này quét góc điều chỉnh theta của bộ lọc Gabor liên tục từ 0 đến 180 độ, tính lại phản hồi với cả bốn thanh ngay tại mỗi thời điểm. Hãy quan sát thanh được tô sáng chuyển đổi khi theta quét qua hướng của từng thanh &mdash; phiên bản động của ma trận phản hồi trội đường chéo đã tính ở trên.

<img src="../animations/gabor_orientation_demo.gif" alt="Ảnh động quét hướng bộ lọc Gabor, với thanh khớp với góc điều chỉnh hiện tại được tô sáng và biểu đồ cột phản hồi của từng thanh cập nhật trực tiếp" style="display: block; margin: 0 auto; max-width: 680px;">

Ma trận phản hồi trội đường chéo rõ rệt: phản hồi lớn nhất của mỗi bộ lọc rơi đúng vào thanh mà nó được điều chỉnh, chính là tính chọn lọc theo hướng mà Hubel và Wiesel đã quan sát được về mặt sinh học.

### Bài tập (Exercises)
1. Thử `db4` (`pywt.Wavelet('db4')`) thay vì `db2` cho phép phân rã ảnh 2D. Diện mạo các dải con LH/HL/HL thay đổi thế nào, khi `db4` có các bộ lọc dài hơn, mượt hơn?
2. Áp dụng `pywt.dwt2` lần thứ hai lên dải con `LL` từ phép phân rã ảnh ở trên, để có tầng thứ hai, thô hơn &mdash; đây là một **kim tự tháp wavelet (wavelet pyramid)**, tương tự trực tiếp với các kim tự tháp Gauss/Laplace ở Bài 10 và 12.
3. Tăng `lambd` (bước sóng của hình sin) trong kernel Gabor trong khi giữ `sigma` cố định. Số sọc nhìn thấy bên trong đường bao Gauss thay đổi thế nào, và bạn kỳ vọng điều đó làm thay đổi tỉ lệ kết cấu ảnh thật mà bộ lọc phản hồi ra sao?